In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
from glob import glob 

import matplotlib.pyplot as plt
import geopandas as gpd
from shapely.geometry import Point
import contextily as cx



### Collect the hourly accumulation for every site and hour from 2024 through 2026

In [ ]:
# Takes about 30-60 sec to run
# Setup your file list and a place to store results
# Use 2024 and 2025 b/c didn't have labeled data from those dates
dir2024 = glob("/home/csutter/NYSM/netcdf/proc/2024/*/*")
print(len(dir2024)) # leap year, 366 days
dir2025 = glob("/home/csutter/NYSM/netcdf/proc/2025/*/*")
print(len(dir2025))
dir2026 = glob("/home/csutter/NYSM/netcdf/proc/2026/*/*")
print(len(dir2026))
files = dir2024+dir2025+dir2026
print(len(files))

results = []

for filename in files:
    ds = xr.open_dataset(filename)
    
    # 1. Get the "Standard" Hourly indices (00:00, 01:00, ..., 23:00)
    hour_mask = ds.time_5M.dt.minute == 0
    hourly_data = ds.isel(time_5M=hour_mask)
    
    # 2. Grab the 00:05 snapshot specifically
    # This is our "Safe Start" to avoid midnight resets
    start_0005 = ds.sel(time_5M=ds.time_5M.dt.hour == 0).isel(time_5M=1) # The 2nd entry (00:05)
    
    station_names = ds.station.values
    times = hourly_data.time_5M.values # [00:00, 01:00, 02:00...]

    # 3. Loop through the hours
    # We skip t_idx 0 (00:00) because we can't calculate a delta for it 
    # (it IS the starting point)
    for t_idx in range(1, len(times)):
        timestamp = times[t_idx]
        ts_str = pd.to_datetime(timestamp).strftime('%Y-%m-%d %H:%M')
        
        # Determine our "Previous" value
        if t_idx == 1:
            # For the 01:00 AM delta, subtract the 00:05 value
            prev_snow = start_0005['snow_depth'].values
            prev_precip = start_0005['precip'].values
        else:
            # For all other hours, use the previous top-of-the-hour
            prev_snow = hourly_data['snow_depth'].isel(time_5M=t_idx-1).values
            prev_precip = hourly_data['precip'].isel(time_5M=t_idx-1).values
            
        current_snow = hourly_data['snow_depth'].isel(time_5M=t_idx).values
        current_precip = hourly_data['precip'].isel(time_5M=t_idx).values
        current_tair = hourly_data['tair'].isel(time_5M=t_idx).values
        
        # Calculate Deltas
        snow_deltas = current_snow - prev_snow
        precip_deltas = current_precip - prev_precip
        
        for s_idx, station_id in enumerate(station_names):
            results.append({
                'timestamp': ts_str,
                'station': station_id,
                'snow_delta': float(snow_deltas[s_idx]),
                'precip_delta': float(precip_deltas[s_idx]),
                'tair': float(current_tair[s_idx])
            })
            
    ds.close()

In [ ]:
# Convert to dataframe

df = pd.DataFrame(results)
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['snow_delta'] = pd.to_numeric(df['snow_delta'])
df['precip_delta'] = pd.to_numeric(df['precip_delta'])
df['date_only'] = pd.to_datetime(df['timestamp']).dt.date

print(len(df))

print(92701*24) # roughly matching the hourly dataset as expected

In [ ]:
df.head(4)


In [ ]:
# df[((df["station"]=="DELE")&(df["date_only"]==pd.to_datetime("2024-12-13").date()))]

In [ ]:
# Only want to look for winter months
# Dec (12), Jan (1), Feb (2), Mar (3)
winter_months = [12, 1, 2, 3]

#  Filter the DataFrame
# .dt.month extracts the month number from the datetime object
winter_df = df[df['timestamp'].dt.month.isin(winter_months)]
winter_df.head(5)

### Find top datetimes for each station, accounting for the fact that we want dates to be spread out (i.e. not all 20 highest precip datetimes being from the same 3 dates)

Way 1 - Using this method
- Find top 20 datetimes for each station while also cross checking that model pred files exist

In [ ]:
# 1 - grab all model preds that have been ran

# alldirs_data_preds = glob("/home/csutter/DRIVE-clean/operational_runs_QPEdata/data_6_ensembling") #HERE!! Where model pred data lives

alldirs_data_preds = glob("/home/csutter/DRIVE-clean/operational_runs/*/data_6_ensembling") #HERE!! Where model pred data lives

# grab ALL model pred file paths
allfiles_data = []
for i in alldirs_data_preds:
    fs = glob(f"{i}/*/*/*/*/*")
    allfiles_data.extend(fs)

# Pre-parse the times into a dictionary
# The dict maps '20240217_1430' -> '/full/path/to/file.csv'
# The dict is used to search for the file path given a date that is needed for an event (Loop 3)
file_lookup_dict = {}
for f in allfiles_data:
    beg = f.rfind("/")
    time_key = f[beg-13:beg]
    # log all string times of interest with the corresponding model pred file in dictionary
    file_lookup_dict[time_key] = f


# def get_weather_regimes_with_models(dataframe, column, file_lookup, mode='high_snow', n_per_station=20):
#     """
#     mode: 'high_snow', 'high_precip', or 'dry'
#     """
#     df_temp = dataframe.dropna(subset=[column]).copy() # drop nan values of the cols we're looking at
    
#     # Pre-check: format timestamp and filter for available model files
#     df_temp['timestamp_formatted'] = df_temp['timestamp'].dt.strftime('%Y%m%d_%H%M')
#     df_temp = df_temp[df_temp['timestamp_formatted'].isin(file_lookup.keys())]

#     if mode == 'high_snow':
#         # Logic: Highest positive snow accumulation
#         # Usually implies tair <= 0, but we let the data speak for itself
#         sorted_df = df_temp.sort_values('snow_delta', ascending=False)
#         unique_events = sorted_df.drop_duplicates(['station', 'date_only'])

#     elif mode == 'high_precip':
#         # Logic: Highest liquid precipitation (Rain Guard: tair > 2.0)
#         df_temp = df_temp[df_temp['tair'] > 4.4] # over 40F
#         sorted_df = df_temp.sort_values('precip_delta', ascending=False)
#         unique_events = sorted_df.drop_duplicates(['station', 'date_only'])

#     elif mode == 'dry':

#         # Logic: Find days where the sum of absolute hourly changes is lowest
#         # This identifies the 24-hour periods with the least amount of movement.
#         df_temp['daily_total_precip'] = df_temp.groupby(['station', 'date_only'])['precip_delta'].transform(lambda x: x.abs().sum()) # looks at the sum of all your hourly deltas across the entire day
        
#         # Identify the unique days for each station, sorted by lowest total precip first
#         unique_days = df_temp.sort_values('daily_total_precip', ascending=True).drop_duplicates(['station', 'date_only'])
        
#         # Take the top N driest days for each station
#         driest_days_summary = unique_days.groupby('station').head(n_per_station).copy()
        
#         # Pick one random hour from each of those identified dry days
#         dry_indices = []
#         for _, row in driest_days_summary.iterrows():
#             day_hours = df_temp[(df_temp['station'] == row['station']) & 
#                                 (df_temp['date_only'] == row['date_only'])]
#             dry_indices.append(day_hours.sample(1, random_state=42))
            
#         unique_events = pd.concat(dry_indices)


#     # Final Sampling: Top N unique-day events per station
#     final_sample = unique_events.groupby('station').head(n_per_station).copy()
#     final_sample['modelpred_file'] = final_sample['timestamp_formatted'].map(file_lookup)
        
#     return final_sample.reset_index(drop=True)


# def get_weather_regimes_with_models(dataframe, column, file_lookup, mode='high_snow', n_per_station=20, preferred_max_per_day=4):
#     df_temp = dataframe.copy()
    
#     # 1. File availability check
#     df_temp['timestamp_formatted'] = df_temp['timestamp'].dt.strftime('%Y%m%d_%H%M')
#     df_temp = df_temp[df_temp['timestamp_formatted'].isin(file_lookup.keys())]

#     # 2. Mode-specific filtering
#     if mode == 'high_snow':
#         # Apply your ~1-inch floor (0.025m)
#         active_df = df_temp[df_temp['snow_delta'] > 0.025].sort_values('snow_delta', ascending=False)
#     elif mode == 'high_precip':
#         # Rain Guard and positive precip
#         active_df = df_temp[(df_temp['tair'] > 4.4) & (df_temp['precip_delta'] > 0)].sort_values('precip_delta', ascending=False)
#     elif mode == 'dry':
#         # Dry logic remains a special case (we don't need the soft-cap here usually)
#         df_temp['daily_total_precip'] = df_temp.groupby(['station', 'date_only'])['precip_delta'].transform(lambda x: x.abs().sum())
#         unique_days = df_temp.sort_values('daily_total_precip', ascending=True).drop_duplicates(['station', 'date_only'])
#         driest_days = unique_days.groupby('station').head(n_per_station)
        
#         dry_samples = []
#         for _, row in driest_days.iterrows():
#             day_hours = df_temp[(df_temp['station'] == row['station']) & (df_temp['date_only'] == row['date_only'])]
#             dry_samples.append(day_hours.sample(1, random_state=42))
#         return pd.concat(dry_samples).reset_index(drop=True)

#     # 3. The "Soft Cap" Logic for Snow/Precip
#     final_indices = []
    
#     for station, station_data in active_df.groupby('station'):
#         # Pass 1: Take up to 4 best hours per day
#         pass1 = station_data.groupby('date_only').head(preferred_max_per_day)
        
#         # If Pass 1 already gives us enough, take the top N
#         if len(pass1) >= n_per_station:
#             final_indices.append(pass1.head(n_per_station))
#         else:
#             # Pass 2: We need more. Take the hours we skipped in Pass 1
#             remaining_data = station_data.drop(pass1.index)
#             needed = n_per_station - len(pass1)
            
#             # Combine Pass 1 with the next best available hours
#             combined = pd.concat([pass1, remaining_data.head(needed)])
#             final_indices.append(combined)

#     unique_events = pd.concat(final_indices) if final_indices else pd.DataFrame()

#     # 4. Map the model files
#     unique_events['modelpred_file'] = unique_events['timestamp_formatted'].map(file_lookup)
        
#     return unique_events.reset_index(drop=True)

def get_weather_regimes_with_models(dataframe, column, file_lookup, mode='high_snow', snow_thresh_m=0.0508, rain_thresh_mm=10.0):
    
    """
    mode: 'high_snow', 'high_precip', or 'dry'
    """

    df_temp = dataframe.copy()
    df_temp['timestamp_formatted'] = df_temp['timestamp'].dt.strftime('%Y%m%d_%H%M')
    df_temp = df_temp[df_temp['timestamp_formatted'].isin(file_lookup.keys())]

    active_col = 'snow_delta' if 'snow' in mode else 'precip_delta'
    df_temp = df_temp.dropna(subset=[active_col]).copy()

    if mode == 'high_snow':
        unique_events = df_temp[df_temp['snow_delta'] >= snow_thresh_m].sort_values('snow_delta', ascending=False)

    elif mode == 'high_precip':
        # Rain Guard (> 4.4C) and using the mm/hr threshold
        unique_events = df_temp[(df_temp['tair'] > 4.4) & (df_temp['precip_delta'] >= rain_thresh_mm)].sort_values('precip_delta', ascending=False)

    elif mode == 'dry':
        # Grab whole days with essentially no precip (b/c we don't want to find some random one hour on a date that rained the remaining 23 hours)
        df_temp['daily_total_precip'] = df_temp.groupby(['station', 'date_only'])['precip_delta'].transform(lambda x: x.abs().sum())
        # Also grab days with little to no snow melt (runoff / wet roads)
        # Add a check for the maximum hourly snow movement (melt or accumulation) that day
        df_temp['daily_max_snow_move'] = df_temp.groupby(['station', 'date_only'])['snow_delta'].transform(lambda x: x.abs().sum())

        dry_days = df_temp[df_temp['daily_total_precip'] < 0.001] # < 0.001mm of precip accumulated in a day, effectively nothing
        dry_days = df_temp[df_temp['daily_max_snow_move'] < 0.01] # < 0.5 inches of snow melt total in a day 

        
        # The Diurnal Snapshot
        # Filter the dry days to only include 00:00, 06:00, 12:00, and 18:00
        target_hours = [6, 18]
        unique_events = dry_days[dry_days['timestamp'].dt.hour.isin(target_hours)]

    unique_events['modelpred_file'] = unique_events['timestamp_formatted'].map(file_lookup)
        
    return unique_events.reset_index(drop=True)


In [ ]:

highsnow = get_weather_regimes_with_models(dataframe = winter_df, column = "snow_delta", file_lookup = file_lookup_dict, mode='high_snow', snow_thresh_m=0.0381) # 1.5 inches / hr


highprecip = get_weather_regimes_with_models(dataframe = winter_df, column = "precip_delta", file_lookup = file_lookup_dict, mode='high_precip', rain_thresh_mm=5) # 5 mm / hr

print(len(highsnow))
print(len(highprecip))

# 1285
# 663

In [ ]:
dry = get_weather_regimes_with_models(dataframe = winter_df, column = "precip_delta", file_lookup = file_lookup_dict, mode='dry') # grab 4 per "dry day", which we designate as less than 0.01 mm/ day. Takes ~20sec

print(len(dry))

In [ ]:
# # --- Usage ---
# # Getting high snow and high precip are fast

# highsnow = get_weather_regimes_with_models(dataframe = winter_df, column = "snow_delta", file_lookup = file_lookup_dict, mode='high_snow', n_per_station=20)

# highprecip = get_weather_regimes_with_models(dataframe = winter_df, column = "precip_delta", file_lookup = file_lookup_dict, mode='high_precip', n_per_station=20)

In [ ]:
# --- Usage ---
# Getting cases takes a little longer, ~ 2-3 mins

# dry = get_weather_regimes_with_models(dataframe = winter_df, column = "precip_delta", file_lookup = file_lookup_dict, mode='dry', n_per_station=20)

Way 2 - NOT using this method but keeping for reference [ Don't need to run ]
- Grabs top 50 datetimes for each stations regardless of whether model pred file exists
- Then maps to pred files that exist or not
- There is a use case for this, but (see commented code second cell down), so keeping for reference code

In [ ]:
import pandas as pd
import numpy as np



def get_varied_extremes(dataframe, column, mode='high', n_per_station=50):
    """
    mode='high': Top positive values.
    mode='low': Values closest to 0 (using absolute value).
    """

    # 1. Drop rows where the target column is NaN and update the variable
    temp_df = dataframe.dropna(subset=[column]).copy()
    
    if mode == 'high':
        # Sort descending to get the largest positive accumulations
        sorted_df = temp_df.sort_values(column, ascending=False)
    
    elif mode == 'low':
        # Create a temporary column for distance from zero
        temp_df['abs_delta'] = temp_df[column].abs()
        # Sort ascending to get values closest to 0
        # We shuffle the data first so we get a variety of dates that hit 0.0
        sorted_df = temp_df.sample(frac=1).sort_values('abs_delta', ascending=True)
    
    # Keep only the most 'extreme' hour for each Date/Station combo
    unique_days = sorted_df.drop_duplicates(['station', 'date_only'])
    
    # Take the top N per station
    final_sample = unique_days.groupby('station').head(n_per_station)
    
    # Clean up the temporary column if it exists
    if 'abs_delta' in final_sample.columns:
        final_sample = final_sample.drop(columns=['abs_delta'])
        
    return final_sample.reset_index(drop=True)

# --- Generate your specific datasets ---

# 1. Highs (Looking for max positive accumulation)
high_snow = get_varied_extremes(winter_df, 'snow_delta', mode='high')
high_precip = get_varied_extremes(winter_df, 'precip_delta', mode='high')

# 2. Lows (Looking for values closest to 0.0)
low_snow = get_varied_extremes(winter_df, 'snow_delta', mode='low')
low_precip = get_varied_extremes(winter_df, 'precip_delta', mode='low')

In [ ]:
# To add info about whether model pred file exists (or not) AFTER summarizing top dates. This code does it after the fact (i.e., top 50 files (above) then tie in whether there is model pred data) 
# This code is for REFERENCE only! B/c we ultimately incorporated it into the date selections.
# But, there is a use case for this. E.g. if we want to find datetimes of high precip that don't have model pred file, and we may want to go back and run it
# We ultimately chose to incorporate the file lookup in the process of selecting the top 50 (rather than after), however, keeping all of this for reference given the usability in previous bullet point. 

# We may not want to use all 50 from each station, maybe just 20, but for each datetime in the highs and lows dfs, see whether we have model run files... from that subset of options that we do have, we can filter to top 20 (or whatever we want)

# 1 - grab all model preds that have been ran

alldirs_data_preds = glob("/home/csutter/DRIVE-clean/operational_runs_QPEdata/data_6_ensembling") #HERE!! Where model pred data lives

# grab ALL model pred file paths
allfiles_data = []
for i in alldirs_data_preds:
    fs = glob(f"{i}/*/*/*/*/*")
    allfiles_data.extend(fs)

# Pre-parse the times into a dictionary
# The dict maps '20240217_1430' -> '/full/path/to/file.csv'
# The dict is used to search for the file path given a date that is needed for an event (Loop 3)
file_lookup = {}
for f in allfiles_data:
    beg = f.rfind("/")
    time_key = f[beg-13:beg]
    # log all string times of interest with the corresponding model pred file in dictionary
    file_lookup[time_key] = f


# 2 - for every datetime in high_snow (all 4 high/low dfs) see which have files that exist

# first need to add a col in the right format
high_snow['timestamp_formatted'] = high_snow['timestamp'].dt.strftime('%Y%m%d_%H%M')

matched_cnn_file = []
matched_cnn_time = []

for etime in high_snow["timestamp_formatted"]:
    if etime in file_lookup: # check the dict we made
        matched_cnn_time.append(etime)
        matched_cnn_file.append(file_lookup[etime])
    else:
        matched_cnn_time.append("no_file")
        matched_cnn_file.append("no_file")
            
# print(len(np.unique(high_snow["timestamp_formatted"])))
print(len(matched_cnn_file))
print(len(high_snow))

high_snow["modelpred_time"] = matched_cnn_time
high_snow["modelpred_file"] = matched_cnn_file

### Gather results for investigation 1

Connect to camera lat and lons

In [ ]:
# Read in mapping dataset NYSM <-> Cams which already exists

nysm_cam_mapping = pd.read_csv("/home/csutter/DRIVE/site_analysis/_reference/ny511sites_closest_mesonet.csv")  # These were only active cameras that we have mapped (2371 total). 

display(nysm_cam_mapping.head(4))
print(len(nysm_cam_mapping))

# see the minimum distance from the closest (min distance) cam to each station
dists = nysm_cam_mapping.groupby(["station"])['distance_km'].agg(min)

# See the avg distance of cams to nearest NYSM station
print(np.mean(nysm_cam_mapping["distance_km"]))

# Active cams are only situated near 77 unique NYSM stations
print("number of stations that serve as a cameras closest station:")
print(len(np.unique(nysm_cam_mapping["station"]))) 



# # Subset to only consider cams within 2 km (want to "verify" on NYSM events but only for cams where it's reasonable to expect verification due to proximity)


nysm_cam_mapping_5km = nysm_cam_mapping[nysm_cam_mapping["distance_km"]<=5]
print("number of cams that have a NYSM within 5km:")
print(len(nysm_cam_mapping_5km))
print("number of stations that have at least one cam within 5km:")
print(len(np.unique(nysm_cam_mapping_5km["station"])))
print(np.unique(nysm_cam_mapping_5km["station"]))

nysm_cam_mapping_2km = nysm_cam_mapping[nysm_cam_mapping["distance_km"]<=2]
print("number of cams that have a NYSM within 2km:")
print(len(nysm_cam_mapping_2km))
print("number of stations that have at least one cam within 2km:")
print(len(np.unique(nysm_cam_mapping_2km["station"])))
print(np.unique(nysm_cam_mapping_2km["station"]))


Using the top snow, top precip, dry (found at top of notebook), see what the models predicted

In [ ]:
# datasets of interest

# dfs with snow events for any NYSM station
print("unique instances from NYSM snow df considering *any* station")
print(len(highsnow))
print(len(highprecip))
print(len(dry))

# Subset to only the stations that have camera sites within 5km
stations_with_cams5km = nysm_cam_mapping_5km["station"]
print(len(np.unique(stations_with_cams5km))) # 19 stations

snowdf = highsnow[highsnow["station"].isin(stations_with_cams5km)].reset_index()
precipdf = highprecip[highprecip["station"].isin(stations_with_cams5km)].reset_index()
drydf = dry[dry["station"].isin(stations_with_cams5km)].reset_index()

print("unique cases from final snow df tied to 5km stations")
print(len(snowdf))
print(len(precipdf))
print(len(drydf))

print("Ex: unique stations from snow")
# print(len(np.unique(highsnow["station"])))
# print(np.unique(highsnow["station"])) # Removed: BKLN, ELMI, MANH

print(len(np.unique(snowdf["station"])))
print(np.unique(snowdf["station"])) # Removed: BKLN, ELMI, MANH

print(len(np.unique(nysm_cam_mapping_5km["station"])))
print(np.unique(nysm_cam_mapping_5km["station"]))


To do: double check that BKLN, ELMI, MANH don't have any hours with > 1.5 inches per hour accum of snow FOR WHICH we have model pred files...

In [ ]:
display(snowdf.head(3))
display(drydf.head(3))

In [ ]:
print(len(snowdf))

# To understand why there are so few snow instances (which are station | hour), have to first show WHICH stations are included in the analysis ... not the "good snowy" ones --- i.e. not Buffalo, Rochester, etc. 
# PLOT THE ONES
# SHOW THE DATES AND DATETIMES USED (IN DF BELOW)



# then where does NCEI come into play? Idk, some case studies or something...
# without turning it into an NCEI verification project, maybe tie in the NYSM data and show how we don't have rapid snowfall for the NYSM stations within that NCEI region

In [ ]:
len(snowdf)

In [ ]:
# for each weather instance, which rows are unique by station | datetime, open the model file for that datetime, subset to the cams that are closest to that station, and then collect the model preds
# Takes ~30s to run

def collect_preds(dfinput = snowdf): 
    stations = []
    sites = [] # list of lists
    times = []
    snowamt = []
    precipamt = []
    preds = []

    for i in range(0, len(dfinput)):
        
        # station of interest
        stationofinterest = dfinput["station"][i]

        # find the list sites that correspond to that station
        sitesofinterest_list = list(nysm_cam_mapping_5km[nysm_cam_mapping_5km["station"]==stationofinterest]["site_id"])
        # print(sitesofinterest_list)

        # read the model pred df, which are by date, one file containing all cam sites
        mfilename = dfinput["modelpred_file"][i]

        # model pred data in this df
        mdf = pd.read_csv(mfilename) # our typical model run run which contains preds for all locations (2371) for a given time instance

        # subset to get cams that correspond to that location
        instance_w_preds = mdf[mdf["site"].isin(sitesofinterest_list)]
        # display(instance_w_preds)
        
        # print(len(instance_w_preds)) # subsetted to just the cams that are relevant for this instance (station | datetime )

        # Collect info for tracking 
        timestamp_formatted = dfinput["timestamp_formatted"][i]
        snow_delta = dfinput["snow_delta"][i]
        precip_delta = dfinput["snow_delta"][i]
        preds_dict = instance_w_preds['select'].value_counts().to_dict()

        # Track info
        stations.append(stationofinterest)
        sites.append(sitesofinterest_list)
        times.append(timestamp_formatted)
        snowamt.append(snow_delta)
        precipamt.append(precip_delta)
        preds.append(preds_dict)


    # turn all the tracked info into a dataframe 

    instances_modelpred_results = pd.DataFrame({"stations":stations,"sites":sites,"times":times,"snowamt":snowamt,"precipamt":precipamt,"preds":preds})

    return instances_modelpred_results

In [ ]:
# RUN on: snowdf, precipd, drydf which were made above
# snow_instances_modelpred_results = collect_preds(dfinput = snowdf)
# precip_instances_modelpred_results = collect_preds(dfinput = precipdf)
dry_instances_modelpred_results = collect_preds(dfinput = drydf) # will take up to ~2 min for this one bc many more examples

In [ ]:
import pandas as pd

def summarize_results2(dfinput):

    # 1. Expand 'preds' dicts into columns and join with 'stations'
    preds_expanded = pd.json_normalize(dfinput['preds'])
    preds_expanded['stations'] = dfinput['stations'].values

    # 2. Group by station and sum class counts
    station_summary = preds_expanded.groupby('stations').sum().reset_index()

    # 3. Identify the class columns (all numeric columns at this stage)
    # We do this NOW so the new columns we add later don't accidentally get turned into percentages
    class_cols = station_summary.select_dtypes(include=['number']).columns

    # 4. Calculate row-wise totals to compute percentages
    row_totals = station_summary[class_cols].sum(axis=1)

    # 5. Create percentage columns for each class
    for col in class_cols:
        # Use 0 if the row total is 0 to avoid division by zero errors
        station_summary[f'{col}_%'] = (station_summary[col] / row_totals).fillna(0) * 100

    # =====================================================================
    # NEW STEP: Calculate 'number of instances' and 'number of sites'
    # =====================================================================
    # Extract the lengths of the 'sites' lists (with a safeguard in case of missing/NaN values)
    sites_lengths = dfinput['sites'].apply(
        lambda x: len(x) if isinstance(x, (list, tuple, set)) else 0
    )

    # Group the original dataframe to get our two new metrics
    stats_df = dfinput.assign(sites_len=sites_lengths).groupby('stations').agg(
        number_of_instances=('stations', 'count'), # Count of rows
        number_of_sites=('sites_len', 'mean')      # Average length of 'sites' list
    ).reset_index()

    # Merge these new columns into our main summary table
    station_summary = pd.merge(station_summary, stats_df, on='stations', how='left')
    # =====================================================================

    # 6. Add a final "Total" row at the bottom
    # Sum the class columns (we use class_cols to avoid summing the percentages)
    total_row_values = station_summary[class_cols].sum()
    total_row = pd.DataFrame([total_row_values])
    total_row['stations'] = 'TOTAL'

    # 7. Recalculate percentages for the Total row
    total_count_sum = total_row[class_cols].sum(axis=1).iloc[0]
    for col in class_cols:
        if total_count_sum > 0:
            total_row[f'{col}_%'] = (total_row[col] / total_count_sum) * 100
        else:
            total_row[f'{col}_%'] = 0

    # Add the overall totals for our two new columns
    total_row['number_of_instances'] = len(dfinput)
    total_row['number_of_sites'] = sites_lengths.mean()

    # Append the total row to the main summary
    station_summary = pd.concat([station_summary, total_row], ignore_index=True)

    # Count the total amount of predictions (stations | hour datetime | cams), which is a sum of the cat cols that are predicted. Need the cols summing over to by dynamic since, for example, it's possible to not have any severe snow preds in the NYSM wet weather instances.
    # 1. Define the possible range of cat columns to sum over
    expected_cols = ["snow_severe", "wet", "snow", "dry", "poor_viz"]
    # 2. Filter down to only the columns that actually exist in the dataframe
    existing_cols = [col for col in expected_cols if col in station_summary.columns]
    # 3. Sum across the rows for those specific columns
    station_summary["count_preds"] = station_summary[existing_cols].sum(axis=1)

    # Optional: Display the result
    display(station_summary)

In [ ]:
# RUN on: snow_instances_modelpred_results, precip_instances_modelpred_results, dry_instances_modelpred_results -  which were made after executing the function above
# summarize_results2(dfinput = snow_instances_modelpred_results)
# summarize_results2(dfinput = precip_instances_modelpred_results)

summarize_results2(dfinput = dry_instances_modelpred_results)

In [ ]:
# Look at NYC

print(np.unique(snow_instances_modelpred_results["stations"]))

nyc_nysm = ['SOUT', 'STON', 'WANT', 'BKLN', 'STAT', 'QUEE', 'MANH', 'BRON']

snow_nonnyc = snow_instances_modelpred_results[~snow_instances_modelpred_results["stations"].isin(nyc_nysm)]

print(len(snow_instances_modelpred_results))
print(len(snow_nonnyc))

# Convert the list of dicts to a temporary DF and sum
result_nonnyc = pd.DataFrame(snow_nonnyc['preds'].tolist()).sum()
print(type(result_nonnyc))

# result['Percentage'] = (result['Values'] / df['Values'].sum()) * 100

print(result_nonnyc)

Plots for visualizing colocation of stations and cams - for reference as needed

In [ ]:
# Connect spatial datasets to get stations (and their lat lons) and sites (and their lat lons) and the mapping between them

##### First - read in data

### Cam lat and lons
cam_coords = pd.read_csv("/home/csutter/DRIVE/site_analysis/_reference/ny511sites_ID_latlon.csv")

display(cam_coords.head(3))
print("len 1")
print(len(cam_coords))

### to subset the cams ^ only to those in New York State specifically 

# 1. Get the built-in low-res map
# Note: Newer versions of geopandas may require installing 'geodatasets'
# path = gpd.datasets.get_path('naturalearth_lowres') # Deprecated in newer versions
# world = gpd.read_file(path)

# 1. Get the official US Census Bureau state boundaries (20m resolution is perfect for this)
# This link is a stable, reputable source for US geographic analysis.
census_url = "https://www2.census.gov/geo/tiger/GENZ2022/shp/cb_2022_us_state_20m.zip"
states = gpd.read_file(census_url)

# Filter for New York
ny_data = states[states['NAME'] == 'New York']
ny_boundary = ny_data.geometry.iloc[0]

# ADD A TINY SAFETY BUFFER (Approx 500 meters)
# This prevents "clipping" cameras that are on bridges or right on the shoreline.
# 0.005 degrees is roughly 500-600 meters in NY.
ny_boundary_safe = ny_boundary.buffer(0.005)

# 2. Convert your camera DF to a GeoDataFrame
gdf_cams = gpd.GeoDataFrame(
    cam_coords, 
    geometry=gpd.points_from_xy(cam_coords.Longitude, cam_coords.Latitude), 
    crs="EPSG:4326"
)

# 3. Check which points are INSIDE the official (buffered) NY Polygon
cam_coords['is_in_ny'] = gdf_cams.geometry.within(ny_boundary_safe)

# 4. Filter
print("Number of NY cams identified:")
cam_coords_NYonly = cam_coords[cam_coords['is_in_ny'] == True].copy()
print(len(cam_coords_NYonly))

# 5. Verify the Staten Island site specifically
verification = cam_coords_NYonly[cam_coords_NYonly["site"]=="Skyline_5973"]
print(f"Skyline_5973 check: {'FOUND' if not verification.empty else 'MISSING'}")

### NYSM lat and lons

# Open just the first file to grab the metadata
ds = xr.open_dataset(files[0])

# Extract the arrays
# NYSM usually stores these as variables or coordinates indexed by 'station'
stations = ds['station'].values
lats = ds['lat'].values
lons = ds['lon'].values

# Create the DataFrame
nysm_coords = pd.DataFrame({
    'station': stations,
    'lat': lats,
    'lon': lons
})

# Close the dataset
ds.close()

# Display the first few to verify
display(nysm_coords.head(3))

print(len(nysm_coords))
print(len(np.unique(nysm_coords["station"])))

### Merge the location datasets  (needs to use the mapping df read in in the first cell)

coords_mapped = nysm_cam_mapping.merge(nysm_coords, how = "inner", left_on = "station", right_on = "station").merge(cam_coords_NYonly, how = "inner", left_on = "site_id", right_on = "site")

print("overall lengths")
print(len(nysm_cam_mapping))
print(len(nysm_coords))
print(len(cam_coords_NYonly))

print(len(coords_mapped))

display(coords_mapped.head(6))

print(len(coords_mapped))

print(len(np.unique(coords_mapped["station"])))

coords_5km = coords_mapped[coords_mapped["distance_km"]<=5] # For plotting only the stations and cams that are withing 5km of each other

In [ ]:
print(len(np.unique(coords_5km["site_id"])))
print(len(np.unique(coords_5km["station"])))

subnyc = coords_5km[coords_5km["station"].isin(['BKLN', 'STAT', 'QUEE', 'MANH', 'BRON'])]
print(len(np.unique(subnyc["site_id"])))



In [ ]:
# # Investigation

# nysm_cam_mapping[nysm_cam_mapping["station"]=="STAT"]

# print(len(nysm_cam_mapping))

# stat_df1 = nysm_cam_mapping[nysm_cam_mapping["station"]=="STAT"]

# cam_coords[cam_coords["site"]=="Skyline_5973"]

# cam_coords_NYonly[cam_coords_NYonly["site"]=="Skyline_5973"]

# dd = stat_df1.merge(cam_coords_NYonly, how = "inner", left_on = "site_id", right_on = "site")
 

In [ ]:
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely.geometry import Point
import contextily as cx

def plot_nysm_with_labeled_buffer(df, target_nysm="VOOR", radius_km=5):
    
    # 1. Setup Geometries
    subset = df[df['station'] == target_nysm].copy()
    # DEBUG PRINTING
    print(f"--- Debugging {target_nysm} ---")
    print(f"Subset Shape: {subset.shape}")
    if not subset.empty:
        print(f"Columns available: {subset.columns.tolist()}")
        
    nysm_lon, nysm_lat = subset['lon'].iloc[0], subset['lat'].iloc[0]
    
    # Create GeoDataFrames
    nysm_point = gpd.GeoDataFrame(geometry=[Point(nysm_lon, nysm_lat)], crs="EPSG:4326").to_crs(epsg=3857)
    cam_points = gpd.GeoDataFrame(subset, geometry=[Point(xy) for xy in zip(subset['Longitude'], subset['Latitude'])], crs="EPSG:4326").to_crs(epsg=3857)
    
    # 2. Create Buffer (radius in meters)
    buffer_geom = nysm_point.buffer(radius_km * 1000)

    # 3. Plotting
    fig, ax = plt.subplots(figsize=(12, 12))
    
    # Plot the Buffer
    buffer_geom.plot(ax=ax, color='blue', alpha=0.15, edgecolor='blue', linestyle='--')
    
    # Plot Points
    cam_points.plot(ax=ax, color='red', markersize=150, edgecolor='white', label='Associated Cameras', zorder=5)
    nysm_point.plot(ax=ax, color='black', marker='X', markersize=150, label=f'NYSM: {target_nysm}', zorder=6)
    
    # 4. ADD THE LABEL "5 km buffer"
    # We place it at the top edge of the buffer: Center X, Center Y + 5000m
    center_x = nysm_point.geometry.iloc[0].x
    center_y = nysm_point.geometry.iloc[0].y
    
    ax.text(center_x, center_y + (radius_km * 1000) + 200, '5 km radius', 
            fontsize=12, color='blue', fontweight='bold', ha='center', va='bottom',
            bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=1))

    # 5. Add Road Map
    cx.add_basemap(ax=ax, source=cx.providers.OpenStreetMap.Mapnik)
    
    ax.set_title(f"NYSM {target_nysm} & Associated Cameras")
    ax.legend(loc='upper right')
    ax.set_axis_off() # Removes the bulky coordinate numbers for a cleaner look
    
    plt.show()

# Execute
# plot_nysm_with_labeled_buffer(your_df)

In [ ]:
plot_nysm_with_labeled_buffer(coords_mapped, target_nysm="STAT")

In [ ]:
# plot some examples
plot_nysm_with_labeled_buffer(coords_mapped, target_nysm="VOOR")

In [ ]:
plot_nysm_with_labeled_buffer(coords_mapped, target_nysm="BATA")

In [ ]:
plot_nysm_with_labeled_buffer(coords_mapped, target_nysm="BUFF")

In [ ]:
plot_nysm_with_labeled_buffer(coords_mapped, target_nysm="ONTA")

In [ ]:
plot_nysm_with_labeled_buffer(coords_mapped, target_nysm="QUEE")

In [ ]:
plot_nysm_with_labeled_buffer(coords_mapped, target_nysm="BRON")

In [ ]:
plot_nysm_with_labeled_buffer(coords_mapped, target_nysm="MANH")

In [ ]:
plot_nysm_with_labeled_buffer(coords_mapped, target_nysm="SCHO")

In [ ]:
plot_nysm_with_labeled_buffer(coords_mapped, target_nysm="TULL")

In [ ]:
plot_nysm_with_labeled_buffer(coords_mapped, target_nysm="WANT")

In [ ]:
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely.geometry import Point
import contextily as cx

def plot_all_nysm_and_cameras(df, radius_km=5):
    """
    Plots every unique NYSM station with a 5km buffer and every camera 
    location from the dataframe on a single NYS map.
    """
    # 1. Prepare Camera Points
    cam_gdf = gpd.GeoDataFrame(
        df, 
        geometry=gpd.points_from_xy(df['Longitude'], df['Latitude']), 
        crs="EPSG:4326"
    ).to_crs(epsg=3857)
    
    # 2. Prepare Unique NYSM Station Points
    nysm_unique = df.drop_duplicates(subset=['station'])
    nysm_gdf = gpd.GeoDataFrame(
        nysm_unique, 
        geometry=gpd.points_from_xy(nysm_unique['lon'], nysm_unique['lat']), 
        crs="EPSG:4326"
    ).to_crs(epsg=3857)
    
    # 3. Create Buffers for all stations at once
    nysm_buffers = nysm_gdf.geometry.buffer(radius_km * 1000)

    # 4. Plotting
    fig, ax = plt.subplots(figsize=(15, 15))
    
    nysm_buffers.plot(ax=ax, color='blue', alpha=0.1, edgecolor='blue', linewidth=0.5)
    cam_gdf.plot(ax=ax, color='red', markersize=50, alpha=0.6, label='Camera Sites', zorder=3)
    nysm_gdf.plot(ax=ax, color='black', marker='+', markersize=100, label='NYSM Stations', zorder=4)
    
    # --- NEW LOGIC: Lock the map extent to New York State ---
    # Define rough bounding box for NYS in Lat/Lon (EPSG:4326)
    nys_bounds = gpd.GeoSeries([
        Point(-79.8, 40.5),  # Southwest corner (near Erie / Staten Island)
        Point(-71.8, 45.1)   # Northeast corner (near Montauk / Canada)
    ], crs="EPSG:4326").to_crs(epsg=3857) # Project to match the plot
    
    # Extract the projected coordinates and set limits
    ax.set_xlim(nys_bounds.geometry.x.iloc[0], nys_bounds.geometry.x.iloc[1])
    ax.set_ylim(nys_bounds.geometry.y.iloc[0], nys_bounds.geometry.y.iloc[1])
    # --------------------------------------------------------

    # 5. Add Basemap
    # Note: Because we set the limits *before* calling contextily, 
    # it knows to download the background tiles for the whole state!
    cx.add_basemap(ax=ax, source=cx.providers.CartoDB.Positron)
    
    # Formatting
    ax.set_title(f"Statewide NYSM Stations & Camera Network", fontsize=16)
    ax.legend(loc='lower right', frameon=True, facecolor='white')
    ax.set_axis_off()
    
    plt.tight_layout()
    plt.show()


# To run:
# plot_all_nysm_and_cameras(coords_mapped)
plot_all_nysm_and_cameras(coords_5km) # For plotting only those that are within 5-km

In [ ]:
# For each nysm station, collect the list of sites we should consider

# Subset to be within 2km
station_listcams = nysm_cam_mapping.groupby(["station"])['site_id'].agg(list).reset_index()

station_listcams

# Reference: Code to investigate nans

In [ ]:
# investigate nans (which we've filtered out in the above function)
# Looks like sometimes the snow depth gauge isn't working. Seemed to be the case for like all of October 2024 for BEAC

d_ex = xr.open_dataset('/home/csutter/NYSM/netcdf/proc/2024/10/20241014.nc')
beac = d_ex.sel(station = 'BEAC')
beac_first = beac.isel(time_5M=287)
beac_last = beac.isel(time_5M=-1)

beac_snow_first = beac_first['snow_depth'].item()
beac_snow_last = beac_last['snow_depth'].item()

print(beac_snow_first)
print(beac_snow_last)

# Reference: Code to investigate outliers
- Code is just for reference. We fixed the outliers by using 00:05 for the first reading of the day rather than 00:00 which contains the accumulation from the previous day, and it messed up the precip deltas

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import pandas as pd
import os

def investigate_case(timestamp_str, target_station, var_name='precip'):
    # 1. Parse the timestamp
    ts = pd.to_datetime(timestamp_str)
    year  = ts.strftime('%Y')
    month = ts.strftime('%m')
    date_str = ts.strftime('%Y%m%d') # Result: '20241014'
    
    # 2. Build the exact path based on your structure
    # Path: /home/csutter/NYSM/netcdf/proc/YYYY/MM/YYYYMMDD.nc
    file_path = f"/home/csutter/NYSM/netcdf/proc/{year}/{month}/{date_str}.nc"
    
    if not os.path.exists(file_path):
        print(f"File not found: {file_path}")
        return

    # 3. Open and Plot
    ds = xr.open_dataset(file_path)
    
    try:
        station_data = ds[var_name].sel(station=target_station)
        
        plt.figure(figsize=(12, 5))
        station_data.plot(marker='.', linestyle='-', linewidth=0.5, color='crimson')
        
        # Mark the hour of the anomaly
        plt.axvline(ts, color='black', linestyle='--', label=f'Outlier Hour: {ts.strftime("%H:%M")}')
        
        plt.title(f"Detailed Analysis: {target_station} | {date_str} | {var_name}")
        plt.ylabel(f"Value ({ds[var_name].attrs.get('units', 'units')})")
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.show()

        # Print the values around that hour to see the "Jump"
        print(f"\n--- 5-Minute Values around {ts.strftime('%H:%M')} ---")
        window = station_data.sel(time_5M=slice(ts - pd.Timedelta(minutes=30), ts + pd.Timedelta(minutes=30)))
        print(window.to_series())

    finally:
        ds.close()

# Test with your specific case
investigate_case("2024-10-14 01:00", "ADDI", var_name='precip')

In [ ]:
import xarray as xr
import pandas as pd
import os

def investigate_raw_values(timestamp_str, target_station, var_name='precip'):
    ts = pd.to_datetime(timestamp_str)
    year, month, date_str = ts.strftime('%Y'), ts.strftime('%m'), ts.strftime('%Y%m%d')
    file_path = f"/home/csutter/NYSM/netcdf/proc/{year}/{month}/{date_str}.nc"
    
    if not os.path.exists(file_path):
        print(f"File not found: {file_path}")
        return

    ds = xr.open_dataset(file_path)
    station_data = ds[var_name].sel(station=target_station)

    # 1. Print the absolute first value of the day (Midnight)
    first_val = station_data.isel(time_5M=0).values
    print(f"--- Midnight (00:00) Raw Value: {first_val} ---")

    # 2. Print the 5-minute raw values around your flagged hour
    print(f"\n--- Raw 5-Min {var_name} Values around {ts.strftime('%H:%M')} ---")
    
    # Looking from the top of the previous hour to the flagged hour
    start_look = ts - pd.Timedelta(hours=1)
    window = station_data.sel(time_5M=slice(start_look, ts))
    
    # Convert to series for a clean timestamped list
    print(window.to_series())

    ds.close()

# Investigation
investigate_raw_values("2024-10-14 01:00", "ADDI", var_name='precip')